# Анализ распределений целевых лог-доходностей

Ноутбук загружает train-split `features/unified_dataset/splits/unified_dataset_train.parquet` из S3 и анализирует целевые переменные:

- `target_log_return_10m`
- `target_log_return_20m`
- `target_log_return_30m`

Для каждого горизонта строятся гистограмма + KDE, нормальное распределение поверх эмпирики, QQ-plot, таблица моментов и Jarque-Bera p-value. Затем подбираются Normal, Student-t, Laplace и GED, после чего модели сравниваются по log-likelihood, AIC и BIC.

In [ ]:
from __future__ import annotations

import io
import math
import sys
from dataclasses import dataclass

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "build_price_feature_day.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from build_price_feature_day import make_s3_client

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

plt.style.use("seaborn-v0_8-whitegrid")

## Конфигурация

`FIT_SAMPLE_SIZE` ограничивает число наблюдений для оценки параметров Student-t и GED. Если нужно считать строго по всему train-сету, установите `FIT_SAMPLE_SIZE = None`.

In [ ]:
S3_BUCKET = "binance-data-downloader"
TRAIN_KEY = "features/unified_dataset/splits/unified_dataset_train.parquet"

TARGET_COLUMNS = {
    "10 мин": "target_log_return_10m",
    "20 мин": "target_log_return_20m",
    "30 мин": "target_log_return_30m",
}

RANDOM_SEED = 42
PLOT_SAMPLE_SIZE = 250_000
FIT_SAMPLE_SIZE = 500_000
HIST_BINS = 180
KDE_GRID_SIZE = 700

rng = np.random.default_rng(RANDOM_SEED)

## Загрузка данных

In [ ]:
def read_s3_parquet(bucket: str, key: str) -> pd.DataFrame:
    s3 = make_s3_client()
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body), columns=list(TARGET_COLUMNS.values()))


raw_targets = read_s3_parquet(S3_BUCKET, TRAIN_KEY)
raw_targets.info(memory_usage="deep")
raw_targets.head()

In [ ]:
missing_columns = [column for column in TARGET_COLUMNS.values() if column not in raw_targets.columns]
if missing_columns:
    raise ValueError(f"Missing target columns: {missing_columns}")

targets = raw_targets.replace([np.inf, -np.inf], np.nan).dropna(axis=0, how="any").copy()
for column in targets.columns:
    targets[column] = pd.to_numeric(targets[column], errors="raise").astype("float64")

print(f"Raw rows: {len(raw_targets):,}")
print(f"Complete finite rows: {len(targets):,}")
print(f"Dropped rows: {len(raw_targets) - len(targets):,}")

## Описательная статистика и Jarque-Bera

In [ ]:
def finite_values(series: pd.Series) -> np.ndarray:
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype="float64")
    return values[np.isfinite(values)]


def sample_values(values: np.ndarray, sample_size: int | None) -> np.ndarray:
    if sample_size is None or len(values) <= sample_size:
        return values.copy()
    idx = rng.choice(len(values), size=sample_size, replace=False)
    return values[idx]


summary_rows = []
target_arrays: dict[str, np.ndarray] = {}

for horizon, column in TARGET_COLUMNS.items():
    values = finite_values(targets[column])
    target_arrays[horizon] = values
    jb_statistic, jb_pvalue = stats.jarque_bera(values)
    summary_rows.append(
        {
            "Горизонт": horizon,
            "N": len(values),
            "Mean": values.mean(),
            "Std": values.std(ddof=1),
            "Skewness": stats.skew(values, bias=False),
            "Kurtosis": stats.kurtosis(values, fisher=True, bias=False),
            "JB p-value": jb_pvalue,
        }
    )

summary = pd.DataFrame(summary_rows)
summary

## Гистограмма + KDE + нормальное распределение

In [ ]:
def robust_grid(values: np.ndarray, quantile_low: float = 0.001, quantile_high: float = 0.999) -> np.ndarray:
    low, high = np.quantile(values, [quantile_low, quantile_high])
    if low == high:
        low, high = values.min(), values.max()
    return np.linspace(low, high, KDE_GRID_SIZE)


fig, axes = plt.subplots(1, 3, figsize=(20, 5), constrained_layout=True)

for axis, (horizon, values) in zip(axes, target_arrays.items()):
    plot_values = sample_values(values, PLOT_SAMPLE_SIZE)
    grid = robust_grid(plot_values)
    mean = plot_values.mean()
    std = plot_values.std(ddof=1)
    kde = stats.gaussian_kde(plot_values)

    axis.hist(plot_values, bins=HIST_BINS, density=True, alpha=0.35, color="#4C78A8", label="Histogram")
    axis.plot(grid, kde(grid), color="#F58518", linewidth=2.0, label="KDE")
    axis.plot(grid, stats.norm.pdf(grid, loc=mean, scale=std), color="#54A24B", linewidth=2.0, label="Normal fit")
    axis.set_title(f"{horizon}: {TARGET_COLUMNS[horizon]}")
    axis.set_xlabel("Log-return")
    axis.set_ylabel("Density")
    axis.legend()

plt.show()

## QQ-plot против нормального распределения

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

for axis, (horizon, values) in zip(axes, target_arrays.items()):
    plot_values = np.sort(sample_values(values, PLOT_SAMPLE_SIZE))
    mean = plot_values.mean()
    std = plot_values.std(ddof=1)
    probabilities = (np.arange(1, len(plot_values) + 1) - 0.5) / len(plot_values)
    theoretical = stats.norm.ppf(probabilities, loc=mean, scale=std)

    axis.scatter(theoretical, plot_values, s=4, alpha=0.25, color="#4C78A8")
    low = min(theoretical.min(), plot_values.min())
    high = max(theoretical.max(), plot_values.max())
    axis.plot([low, high], [low, high], color="#E45756", linewidth=2)
    axis.set_title(f"QQ-plot: {horizon}")
    axis.set_xlabel("Normal theoretical quantiles")
    axis.set_ylabel("Empirical quantiles")

plt.show()

## Подбор распределений и сравнение AIC/BIC

In [ ]:
@dataclass(frozen=True)
class DistributionSpec:
    name: str
    scipy_distribution: object
    parameter_count: int


DISTRIBUTIONS = (
    DistributionSpec("Normal", stats.norm, 2),
    DistributionSpec("Student-t", stats.t, 3),
    DistributionSpec("Laplace", stats.laplace, 2),
    DistributionSpec("GED", stats.gennorm, 3),
)


def fit_distribution(values: np.ndarray, spec: DistributionSpec) -> dict:
    fit_values = sample_values(values, FIT_SAMPLE_SIZE)
    params = spec.scipy_distribution.fit(fit_values)
    logpdf = spec.scipy_distribution.logpdf(fit_values, *params)
    finite_logpdf = logpdf[np.isfinite(logpdf)]
    if len(finite_logpdf) != len(fit_values):
        raise ValueError(f"{spec.name} produced non-finite logpdf values")

    log_likelihood = float(finite_logpdf.sum())
    n = len(fit_values)
    k = spec.parameter_count
    return {
        "Distribution": spec.name,
        "N fit": n,
        "Params": params,
        "Log-Likelihood": log_likelihood,
        "AIC": 2 * k - 2 * log_likelihood,
        "BIC": k * math.log(n) - 2 * log_likelihood,
    }


fit_rows = []
for horizon, values in target_arrays.items():
    for spec in DISTRIBUTIONS:
        row = fit_distribution(values, spec)
        row["Горизонт"] = horizon
        fit_rows.append(row)

fit_results = pd.DataFrame(fit_rows)
fit_results = fit_results[
    ["Горизонт", "Distribution", "N fit", "Log-Likelihood", "AIC", "BIC", "Params"]
].sort_values(["Горизонт", "AIC"]).reset_index(drop=True)
fit_results

In [ ]:
best_by_aic = fit_results.loc[fit_results.groupby("Горизонт")["AIC"].idxmin()].copy()
best_by_bic = fit_results.loc[fit_results.groupby("Горизонт")["BIC"].idxmin()].copy()

best_comparison = best_by_aic[["Горизонт", "Distribution", "AIC", "BIC", "Log-Likelihood"]].rename(
    columns={"Distribution": "Best by AIC"}
).merge(
    best_by_bic[["Горизонт", "Distribution"]].rename(columns={"Distribution": "Best by BIC"}),
    on="Горизонт",
    how="left",
)
best_comparison

## Визуальная проверка fitted distributions

Графики ниже накладывают все fitted distributions на эмпирическую KDE. Это помогает увидеть, где именно модель выигрывает или проигрывает: центр, плечи или хвосты.

In [ ]:
distribution_lookup = {spec.name: spec.scipy_distribution for spec in DISTRIBUTIONS}
colors = {
    "Normal": "#54A24B",
    "Student-t": "#B279A2",
    "Laplace": "#E45756",
    "GED": "#72B7B2",
}

fig, axes = plt.subplots(1, 3, figsize=(20, 5), constrained_layout=True)

for axis, (horizon, values) in zip(axes, target_arrays.items()):
    plot_values = sample_values(values, PLOT_SAMPLE_SIZE)
    grid = robust_grid(plot_values)
    kde = stats.gaussian_kde(plot_values)
    axis.plot(grid, kde(grid), color="black", linewidth=2.2, label="Empirical KDE")

    horizon_fits = fit_results[fit_results["Горизонт"].eq(horizon)]
    for _, row in horizon_fits.iterrows():
        distribution_name = row["Distribution"]
        distribution = distribution_lookup[distribution_name]
        params = row["Params"]
        axis.plot(
            grid,
            distribution.pdf(grid, *params),
            linewidth=1.8,
            color=colors[distribution_name],
            label=distribution_name,
        )

    axis.set_title(f"Fitted distributions: {horizon}")
    axis.set_xlabel("Log-return")
    axis.set_ylabel("Density")
    axis.legend()

plt.show()